In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
chicago = pd.read_csv('/content/drive/MyDrive/final_csv/crime.csv')

In [4]:
features = [
    'Year', 'Month', 'Primary Type',
    'Description', 'Community Area',
    'Latitude', 'Longitude', 'Offense Level Median',
    'Dist_to_Nearest_HighRisk','min_distance_to_police'

]

target = 'Arrest'

In [5]:
ml_data = chicago[features + [target]].dropna().copy()

In [6]:
# 라벨 인코딩
from sklearn.preprocessing import LabelEncoder
le_dict = {}
for col in features:
    if ml_data[col].dtype == 'object':
        le = LabelEncoder()
        ml_data[col] = le.fit_transform(ml_data[col])
        le_dict[col] = le

In [7]:
# X, y 분리
X_data = ml_data.drop(columns=[target])
y_data = ml_data[target]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data,  stratify=y_data, test_size=0.3, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.3, stratify=y_train,  random_state=42)

In [9]:
from tensorflow.keras import layers, models, regularizers

model = models.Sequential([
    layers.InputLayer(shape=(X_train.shape[1],)),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(1, activation='sigmoid'),
])


In [10]:
# 학습 준비 (optimizer = adam, loss = 문제에 맞추어 지정)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [11]:
from keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',     # 모니터링할 값
    patience=3,             # 성능 개선이 없을 때 몇 번 더 기다릴지
    restore_best_weights=True # 가장 성능 좋았을 때 weight로 복원
)
# 모델 학습 (100번 반복)
model.fit(X_train, y_train, epochs=50, validation_data=(X_val, y_val), callbacks=[early_stop])

Epoch 1/50
126737/126737 ━━━━━━━━━━━━━━━━━━━━ 356s 3ms/step - accuracy: 0.8072 - loss: 0.4825 - val_accuracy: 0.7461 - val_loss: 0.8577
Epoch 2/50
126737/126737 ━━━━━━━━━━━━━━━━━━━━ 352s 3ms/step - accuracy: 0.8115 - loss: 0.4734 - val_accuracy: 0.6395 - val_loss: 0.6417
Epoch 3/50
126737/126737 ━━━━━━━━━━━━━━━━━━━━ 352s 3ms/step - accuracy: 0.8116 - loss: 0.4733 - val_accuracy: 0.7541 - val_loss: 0.5353
Epoch 4/50
126737/126737 ━━━━━━━━━━━━━━━━━━━━ 352s 3ms/step - accuracy: 0.8115 - loss: 0.4734 - val_accuracy: 0.6015 - val_loss: 0.6515
Epoch 5/50
126737/126737 ━━━━━━━━━━━━━━━━━━━━ 351s 3ms/step - accuracy: 0.8111 - loss: 0.4742 - val_accuracy: 0.7461 - val_loss: 1.3878
Epoch 6/50
126737/126737 ━━━━━━━━━━━━━━━━━━━━ 352s 3ms/step - accuracy: 0.8104 - loss: 0.4749 - val_accuracy: 0.6410 - val_loss: 0.6076


In [12]:
# 모델 평가
model.evaluate(X_test, y_test)

77594/77594 ━━━━━━━━━━━━━━━━━━━━ 137s 2ms/step - accuracy: 0.7536 - loss: 0.5362


[0.5363463759422302, 0.7535229325294495]

In [ ]:
# test의 10번째 값의 예측값과 실제값 동일여부 확인
import tensorflow as tf
sample = X_test[10, :].reshape(1,-1)
sample_pred = tf.round(model.predict(sample),0)

if sample_pred == y_test[10]:
    print("정답")
else:
    print("오답")